# AIST-FYP Colab Mitigation Evaluation Notebook

This notebook runs **mitigation module evaluation** (baseline, full pipeline, mitigation-only variants) using existing project scripts.

## Default profile
- Scope: **RAGTruth + CiteEval**
- Variants: **baseline + full pipeline + mitigation-only variants**
- Runtime target: **smoke test**

## 🔑 Setup Colab Secrets

1. Open the **Secrets** panel (key icon) in Colab sidebar.
2. Add secrets as needed:
   - `DEEPSEEK_API_KEY` (recommended for CiteEval cost)
   - `OPENAI_API_KEY` (optional if using OpenAI provider)
   - `HUGGINGFACE_TOKEN` (optional for gated models)
3. Turn on **Notebook access** for each secret.

The next cell loads these into environment variables automatically.

In [ ]:
# Load Colab secrets into environment variables
import os

try:
    from google.colab import userdata
    secret_keys = ["DEEPSEEK_API_KEY", "OPENAI_API_KEY", "HUGGINGFACE_TOKEN"]
    loaded = []
    for key in secret_keys:
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
                loaded.append(key)
        except Exception:
            pass

    if loaded:
        print("Loaded secrets:", ", ".join(loaded))
    else:
        print("No Colab secrets loaded. Add keys via Secrets panel if needed.")
except ImportError:
    print("Not running in Colab. Export API keys manually in your environment.")

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["mitigation"]

RUN_RAGTRUTH_MITIGATION = True
RUN_CITEEVAL_MITIGATION = True

# Shared
STRATEGY = "production"            # development | validation | production
VARIANTS = [
    "baseline",
    "full_pipeline",
    "mitigation_filter_only",
    "mitigation_rerank_only",
    "mitigation_reprompt_only",
]

# RAGTruth mitigation script knobs
RAGTRUTH_SPLIT = "test"
RAGTRUTH_EVAL_MODE = "gold_context_generation"  # ragtruth_eval | normal | gold_context_generation
RAGTRUTH_MAX_SAMPLES = 10            # smoke test; set None for full split
RAGTRUTH_BATCH_SIZE = 10

# LongT5 colab profile knobs
GENERATOR_MODEL = "google/long-t5-tglobal-base"
GENERATOR_MAX_INPUT_TOKENS = 4096
GENERATOR_LOAD_IN_8BIT = True

# CiteEval mitigation script knobs
CITEEVAL_CONTEXT_SOURCE = "retrieval"  # retrieval | oracle
CITEEVAL_SYSTEM_SOURCE = "benchmark/CiteEval/data/system_eval/system_eval_examples.json"
CITEEVAL_ORACLE_DATASET = "asqa"       # asqa | eli5 | msmarco
CITEEVAL_ORACLE_SOURCE = ""            # optional explicit oracle JSONL path
CITEEVAL_MAX_SAMPLES = 10              # smoke test
CITEEVAL_PROVIDER = "deepseek"        # deepseek | openai
CITEEVAL_MODEL_NAME = "deepseek-chat"
CITEEVAL_VERSION = "citeeval-auto-12272024"
CITEEVAL_MODULES = "ca,ce,cr_itercoe,cr_editdist"
CITEEVAL_N_THREADS = 8
CITEEVAL_CITED_ONLY = False

# Artifacts reminder (must exist inside repo folder after clone)
# - data/indexes/{STRATEGY}/faiss.index
# - data/indexes/{STRATEGY}/metadata.pkl
# - data/processed/wiki_chunks_{STRATEGY}.jsonl
# - benchmark/RAGTruth/dataset
# - benchmark/CiteEval
# - benchmark/CiteEval/data/dev/*_oracle.dev.jsonl (for oracle mode)
ARTIFACTS_IN_PROJECT = True

# Persistent storage policy (survive Colab runtime resets)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIST-FYP-colab-outputs"
DRIVE_WORK_ROOT = f"{DRIVE_OUTPUT_DIR}/work_mitigation"
LOCAL_WORK_ROOT = DRIVE_WORK_ROOT
RUN_TAG = "colab_mitigation_eval"
SKIP_EXPORT_COPY_WHEN_PERSISTENT = True

# Persistent runtime output dirs
RAG_MITIGATION_OUTPUT_DIR = f"{LOCAL_WORK_ROOT}/outputs/mitigation_eval"
CITE_MITIGATION_OUTPUT_DIR = f"{LOCAL_WORK_ROOT}/outputs/mitigation_eval_citebench"
CITEEVAL_SYSTEM_OUTPUTS_DIR = f"{LOCAL_WORK_ROOT}/citeeval/system_eval_outputs"
CITEEVAL_METRIC_OUTPUTS_DIR = f"{LOCAL_WORK_ROOT}/citeeval/metric_eval_outputs"
CONFIG_PATH = "config.colab.yaml"

In [ ]:
import os
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

def run(cmd, cwd=None, check=True, stream=True):
    print(f"\n$ {cmd}")
    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout="".join(out_lines),
            stderr=None,
        )
    else:
        completed = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

def exists_or_raise(path, msg):
    if not Path(path).exists():
        raise FileNotFoundError(f"{msg}: {path}")

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def ensure_symlink_dir(link_path: Path, target_path: Path):
    target_path = Path(target_path)
    link_path = Path(link_path)
    ensure_dir(target_path)
    ensure_dir(link_path.parent)

    if link_path.is_symlink():
        current_target = Path(os.readlink(link_path))
        if current_target == target_path:
            return
        link_path.unlink()
    elif link_path.exists():
        if link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()

    os.symlink(target_path, link_path, target_is_directory=True)

def latest_subdir(parent: Path):
    if not parent.exists():
        return None
    dirs = [p for p in parent.iterdir() if p.is_dir()]
    return max(dirs, key=lambda p: p.name) if dirs else None

In [ ]:
# Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

if not str(LOCAL_WORK_ROOT).startswith('/content/drive/'):
    print(f"⚠️ LOCAL_WORK_ROOT is not on Drive: {LOCAL_WORK_ROOT}")
    print("Artifacts may not survive Colab runtime reset.")

ensure_dir(LOCAL_WORK_ROOT)
print("Persistent LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)

if Path(REPO_DIR).exists():
    print(f"Repo dir already exists: {REPO_DIR}")
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}", stream=True)

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR, stream=True)
run("git log -1 --oneline", cwd=REPO_DIR, stream=True)

# Persist runtime outputs via symlink to Drive-backed workspace
symlink_map = {
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval': Path(RAG_MITIGATION_OUTPUT_DIR),
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval_citebench': Path(CITE_MITIGATION_OUTPUT_DIR),
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'system_eval_outputs': Path(CITEEVAL_SYSTEM_OUTPUTS_DIR),
}

for link_path, target_path in symlink_map.items():
    ensure_symlink_dir(link_path, target_path)
    print(f"Persisted path: {link_path} -> {target_path}")

In [ ]:
# Install dependencies
run("python -m pip install -U pip wheel setuptools", stream=True)
run("python -m pip install -U uv", stream=True)

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False, stream=True)

spacy_model_wheel = "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
spacy_install_cmd = "python -m spacy download en_core_web_sm"

if result.returncode == 0:
    uv_python = uv_project / ".venv" / "bin" / "python"
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    spacy_install_cmd = f"uv pip install --python {uv_python} {spacy_model_wheel}"
    print(f"✅ uv sync complete: {uv_project}")
else:
    print('\n⚠️ uv sync failed. Falling back to pip requirements install...')
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f"pip install --extra-index-url {pytorch_index} -r {requirements_path}"
    fallback_result = run(install_cmd, cwd=REPO_DIR, check=False, stream=True)

    if fallback_result.returncode != 0:
        print('\n⚠️ Full requirements install failed. Falling back to Colab-torch-compatible install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = Path(REPO_DIR) / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
        run(f"pip install -r {temp_req}", cwd=REPO_DIR, stream=True)

# spaCy model required by verifier
run(spacy_install_cmd, cwd=REPO_DIR, stream=True)

In [ ]:
# Runtime + env setup
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

os.environ["CITEEVAL_PROVIDER"] = CITEEVAL_PROVIDER
os.environ["CITEEVAL_ROOT"] = str(Path(REPO_DIR) / "benchmark/CiteEval")
extra_paths = [str(Path(REPO_DIR) / "benchmark/CiteEval"), str(Path(REPO_DIR) / "benchmark/CiteEval/src")]
existing_pp = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (existing_pp + os.pathsep if existing_pp else "") + os.pathsep.join(extra_paths)

print("CITEEVAL_PROVIDER =", os.environ.get("CITEEVAL_PROVIDER"))
print("DEEPSEEK_API_KEY set =", bool(os.environ.get("DEEPSEEK_API_KEY")))
print("OPENAI_API_KEY set =", bool(os.environ.get("OPENAI_API_KEY")))

In [ ]:
# Create Colab-specific config with LongT5 settings
import yaml

base_config = Path(REPO_DIR) / 'config.yaml'
colab_config = Path(REPO_DIR) / CONFIG_PATH

with open(base_config, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg.setdefault('processing', {})['device'] = 'cuda'
cfg.setdefault('models', {})['generator'] = GENERATOR_MODEL
cfg.setdefault('generation', {})['max_input_tokens'] = int(GENERATOR_MAX_INPUT_TOKENS)
cfg['generation']['load_in_8bit'] = bool(GENERATOR_LOAD_IN_8BIT)
cfg.setdefault('retrieval', {}).setdefault('faiss', {})['use_gpu'] = False
cfg['retrieval']['faiss']['gpu_id'] = 0
cfg.setdefault('evaluation', {}).setdefault('benchmarks', {}).setdefault('ragtruth', {})['ragtruth_eval_mode'] = RAGTRUTH_EVAL_MODE

with open(colab_config, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print('Wrote', colab_config)
print('Generator model:', cfg['models']['generator'])
print('Max input tokens:', cfg['generation']['max_input_tokens'])
print('Load in 8-bit:', cfg['generation']['load_in_8bit'])

In [ ]:
# Preflight artifact checks
repo = Path(REPO_DIR)
faiss_index = repo / f"data/indexes/{STRATEGY}/faiss.index"
index_meta = repo / f"data/indexes/{STRATEGY}/metadata.pkl"
chunks_file = repo / f"data/processed/wiki_chunks_{STRATEGY}.jsonl"
ragtruth_dataset = repo / "benchmark/RAGTruth/dataset"
citeeval_root = repo / "benchmark/CiteEval"
citeeval_source = repo / CITEEVAL_SYSTEM_SOURCE

exists_or_raise(faiss_index, "Missing FAISS index")
exists_or_raise(index_meta, "Missing index metadata")
exists_or_raise(chunks_file, "Missing processed chunks")
if RUN_RAGTRUTH_MITIGATION:
    exists_or_raise(ragtruth_dataset, "Missing RAGTruth dataset directory")
if RUN_CITEEVAL_MITIGATION:
    exists_or_raise(citeeval_root, "Missing benchmark/CiteEval directory")
    if CITEEVAL_CONTEXT_SOURCE == "oracle":
        if CITEEVAL_ORACLE_SOURCE:
            oracle_path = repo / CITEEVAL_ORACLE_SOURCE
        else:
            oracle_path = repo / f"benchmark/CiteEval/data/dev/{CITEEVAL_ORACLE_DATASET}_oracle.dev.jsonl"
        exists_or_raise(oracle_path, "Missing CiteEval oracle source JSONL")
    else:
        exists_or_raise(citeeval_source, "Missing CiteEval system source JSON")

print("Preflight checks passed.")

In [ ]:
# Run RAGTruth module-level evaluation
if RUN_RAGTRUTH_MITIGATION:
    rag_cmd = [
        "python scripts/evaluate_mitigation_strategy.py",
        f"--config {CONFIG_PATH}",
        f"--split {RAGTRUTH_SPLIT}",
        f"--batch-size {RAGTRUTH_BATCH_SIZE}",
        f"--strategy {STRATEGY}",
        f"--ragtruth-eval-mode {RAGTRUTH_EVAL_MODE}",
        "--variants " + " ".join(VARIANTS),
    ]

    if RAGTRUTH_MAX_SAMPLES is not None:
        rag_cmd.append(f"--max-samples {RAGTRUTH_MAX_SAMPLES}")

    run(" ".join(rag_cmd), cwd=REPO_DIR)
else:
    print("Skipped RAGTruth module-level evaluation.")

In [ ]:
# Run CiteEval module-level evaluation
if RUN_CITEEVAL_MITIGATION:
    cite_cmd = [
        "python scripts/evaluate_mitigation_citebench.py",
        f"--config {CONFIG_PATH}",
        "--dataset-role mitigation",
        f"--strategy {STRATEGY}",
        f"--context-source {CITEEVAL_CONTEXT_SOURCE}",
        "--variants " + " ".join(VARIANTS),
        f"--provider {CITEEVAL_PROVIDER}",
        f"--model-name {CITEEVAL_MODEL_NAME}",
        f"--version {CITEEVAL_VERSION}",
        f"--modules {CITEEVAL_MODULES}",
        f"--n-threads {CITEEVAL_N_THREADS}",
    ]

    if CITEEVAL_CONTEXT_SOURCE == "oracle":
        if CITEEVAL_ORACLE_SOURCE:
            cite_cmd.append(f"--oracle-source {CITEEVAL_ORACLE_SOURCE}")
        else:
            cite_cmd.append(f"--oracle-dataset {CITEEVAL_ORACLE_DATASET}")
    else:
        cite_cmd.append(f"--system-source {CITEEVAL_SYSTEM_SOURCE}")

    if CITEEVAL_MAX_SAMPLES is not None:
        cite_cmd.append(f"--max-samples {CITEEVAL_MAX_SAMPLES}")
    if CITEEVAL_CITED_ONLY:
        cite_cmd.append("--cited-only")

    run(" ".join(cite_cmd), cwd=REPO_DIR)
else:
    print("Skipped CiteEval module-level evaluation.")

In [ ]:
# Locate latest summaries and export manifest/snapshot
repo = Path(REPO_DIR)
rag_root = repo / "outputs/mitigation_eval"
cite_root = repo / "outputs/mitigation_eval_citebench"

latest_rag = latest_subdir(rag_root)
latest_cite_parent = latest_subdir(cite_root)
latest_cite = None
if latest_cite_parent:
    preferred = latest_cite_parent / CITEEVAL_CONTEXT_SOURCE
    latest_cite = preferred if preferred.exists() else latest_subdir(latest_cite_parent)

if latest_rag:
    print("Latest RAGTruth module-eval run:", latest_rag)
    print("-", latest_rag / "summary.json")
    print("-", latest_rag / "summary.md")
else:
    print("No RAGTruth module-eval run found.")

if latest_cite:
    print("Latest CiteEval module-eval run:", latest_cite)
    print("-", latest_cite / "summary.json")
    print("-", latest_cite / "summary.md")
else:
    print("No CiteEval module-eval run found.")

ensure_dir(DRIVE_OUTPUT_DIR)
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_root = Path(DRIVE_OUTPUT_DIR) / f"{RUN_TAG}_{STRATEGY}_{run_stamp}"
ensure_dir(export_root)

local_work_root_path = Path(LOCAL_WORK_ROOT).resolve()
artifacts_on_drive = str(local_work_root_path).startswith('/content/drive/')

targets = [p for p in [latest_rag, latest_cite] if p is not None]
exported_paths = []

if artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT:
    print("Artifacts are already on Drive; skipping duplicate copy.")
    for t in targets:
        exported_paths.append(str(t))
        print("Registered existing artifact:", t)
else:
    for t in targets:
        dest = export_root / t.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(t, dest)
        exported_paths.append(str(dest))
        print("Exported:", t, "->", dest)

manifest = {
    "timestamp": run_stamp,
    "strategy": STRATEGY,
    "variants": VARIANTS,
    "ragtruth_eval_mode": RAGTRUTH_EVAL_MODE,
    "citeeval_context_source": CITEEVAL_CONTEXT_SOURCE,
    "local_work_root": LOCAL_WORK_ROOT,
    "repo_dir": REPO_DIR,
    "export_root": str(export_root),
    "targets": [str(t) for t in targets],
    "exported_artifacts": exported_paths,
    "artifacts_already_on_drive": artifacts_on_drive,
    "skipped_copy_when_persistent": bool(artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT),
}

manifest_path = export_root / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Drive output directory:", DRIVE_OUTPUT_DIR)
print("Manifest:", manifest_path)

## Scale-up after smoke test

- Set `RAGTRUTH_MAX_SAMPLES = None` for full split.
- Increase `CITEEVAL_MAX_SAMPLES` or set to `None`.
- Keep mitigation-only variants in `VARIANTS` (do not add `verifier_*` here).
- Keep one parameter profile per experiment for reproducibility.